In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Try reading the CSV with a specific encoding
try:
    df = pd.read_csv('UpdatedResume_DataSet (1).csv', encoding='utf-8')
except UnicodeDecodeError:
    # If utf-8 fails, try another common encoding
    df = pd.read_csv('UpdatedResume_DataSet (1).csv', encoding='latin-1')

# Display the Frame
df.head(10)

In [ ]:
df.sample(10)


In [ ]:
df.shape

In [ ]:
df['Category'].value_counts()

In [ ]:
plt.figure(figsize=(10,15))
# sns_plot = sns.countplot(x=df['Category'])
sns_plot = sns.countplot(x=df['Category'], palette='hls')

# Increase the size of the x and y labels
sns_plot.set_xlabel('Category')
sns_plot.set_ylabel('Count')

# Increase the size of the tick labels
sns_plot.tick_params(axis='x')
sns_plot.tick_params(axis='y')

plt.xticks(rotation=90)
plt.show()

In [ ]:
counts=df['Category'].value_counts()
labels=df['Category'].unique()
plt.figure(figsize=(12,8))
plt.pie(counts,labels=labels, autopct='%1.1f%%',startangle=60)
plt.show()

In [ ]:
df.columns

In [ ]:
df['Resume'][2]

In [ ]:
import nltk
nltk.download("stopwords")

In [ ]:
import re
import string
from nltk.corpus import stopwords  # Import stopwords from NLTK

def preprocessing(text):
  """
  This function cleans a resume text by removing stopwords and special characters.

  Args:
      text: The text of the resume document.

  Returns:
      A string containing the cleaned resume text.
  """
  # Lowercase the text
  text = text.lower()

  # Remove punctuation
  text = text.translate(str.maketrans('', '', string.punctuation))

  # Remove stopwords (using NLTK stopwords for English)
  stop_words = stopwords.words('english')
  text = ' '.join([word for word in text.split() if word not in stop_words])

  # Remove extra whitespace
  text = ' '.join(text.split())

  return text

# Example usage
resume_text = "This is a sample resume with stopwords and special characters!@#$%^"
cleaned_text = preprocessing(resume_text)
print(cleaned_text)

In [ ]:
import re
def clean_resume(txt):
    txt=preprocessing(txt)
    cleanText = re.sub('http\S+\s', ' ', txt)
    cleanText = re.sub('RT|cc', ' ', cleanText)
    cleanText = re.sub('#\S+\s', ' ', cleanText)
    cleanText = re.sub('@\S+', '  ', cleanText)
    cleanText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', cleanText)
    cleanText = re.sub(r'[^\x00-\x7f]', ' ', cleanText)
    cleanText = re.sub('\s+', ' ', cleanText)
    return cleanText

In [ ]:
df['Resume']=df['Resume'].apply(lambda x: clean_resume(x))
df['Resume']

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [ ]:
le.fit(df['Category'])
df['Category']=le.transform(df['Category'])

In [ ]:
df.Category.unique()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(stop_words='english')
tfidf.fit(df['Resume'])
required_text=tfidf.transform(df['Resume'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test, y_train, y_test=train_test_split(required_text, df['Category'],test_size=0.3,random_state=30)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
# KNN model with OneVsRest strategy
clf=OneVsRestClassifier(KNeighborsClassifier())
clf.fit(X_train,y_train)
y_pred=clf.predict(X_test)
score=accuracy_score(y_test,y_pred)
score

In [ ]:
# Logistic Regression model with OneVsRest strategy
from sklearn.linear_model import LogisticRegression
model = OneVsRestClassifier(LogisticRegression())
model.fit(X_train,y_train)
y_pred=clf.predict(X_test)
score=accuracy_score(y_test,y_pred)
score

In [ ]:
model =LogisticRegression()
model.fit(X_train,y_train)
y_pred=clf.predict(X_test)
score=accuracy_score(y_test,y_pred)
score

In [ ]:
from sklearn.model_selection import cross_val_score

# Logistic Regression cross-validation
logreg_cv_scores = cross_val_score(model, X_train, y_train, cv=5)
print("Logistic Regression CV Accuracy: ", logreg_cv_scores.mean())

# KNN cross-validation
knn_cv_scores = cross_val_score(clf, X_train, y_train, cv=5)
print("KNN CV Accuracy: ", knn_cv_scores.mean())

In [ ]:
from sklearn.model_selection import cross_val_score

# Cross-validation for Logistic Regression
logreg_cv_scores = cross_val_score(model, X_train, y_train, cv=5)
logreg_train_accuracy = model.score(X_train, y_train)
logreg_test_accuracy = model.score(X_test, y_test)

print(f"Logistic Regression - Train Accuracy: {logreg_train_accuracy}")
print(f"Logistic Regression - Test Accuracy: {logreg_test_accuracy}")
print(f"Logistic Regression - Cross-validation Accuracy: {logreg_cv_scores.mean()}")

# Cross-validation for KNN
knn_cv_scores = cross_val_score(clf, X_train, y_train, cv=5)
knn_train_accuracy = clf.score(X_train, y_train)
knn_test_accuracy = clf.score(X_test, y_test)

print(f"KNN - Train Accuracy: {knn_train_accuracy}")
print(f"KNN - Test Accuracy: {knn_test_accuracy}")
print(f"KNN - Cross-validation Accuracy: {knn_cv_scores.mean()}")


In [ ]:
import pickle
pickle.dump(tfidf,open('tfidf.pkl','wb'))
pickle.dump(model, open('clf.pkl', 'wb'))

In [ ]:
myresume =''''Good grasping quality and skillful work Education Details March 2013 to
March 2018 B A LL B Law Solapur Maharashtra Solapur University Advocate Skill Details
Good knowledge of typing as well as many other activities Exprience Less than 1 year months
Company Details company District and Session court of solapur description Forward thinking
individual with refined interpersonal and multitasking skills Looking to join a progressive
organization to provide assistance in Legal work company District and
Session court of solapur description Provide legal assistance in legal work'''

In [ ]:
# Load the trained classifier
clf = pickle.load(open('clf.pkl', 'rb'))
# Clean the input resume
cleaned_resume = clean_resume(myresume)

# Transform the cleaned resume using the trained TfidfVectorizer
input_features = tfidf.transform([cleaned_resume])

# Make the prediction using the loaded classifier
prediction_id = clf.predict(input_features)[0]

# Map category ID to category name
category_mapping = {
    15: "Java Developer",
    23: "Testing",
    8: "DevOps Engineer",
    20: "Python Developer",
    24: "Web Designing",
    12: "HR",
    13: "Hadoop",
    3: "Blockchain",
    10: "ETL Developer",
    18: "Operations Manager",
    6: "Data Science",
    22: "Sales",
    16: "Mechanical Engineer",
    1: "Arts",
    7: "Database",
    11: "Electrical Engineering",
    14: "Health and fitness",
    19: "PMO",
    4: "Business Analyst",
    9: "DotNet Developer",
    2: "Automation Testing",
    17: "Network Security Engineer",
    21: "SAP Developer",
    5: "Civil Engineer",
    0: "Advocate",
}

category_name = category_mapping.get(prediction_id, "Unknown")

print("Predicted Category:", category_name)
print(prediction_id)